# Gemma 4 ALT-v1 — 2-Task Kaggle Reproducibility Smoke Test

Purpose: verify that the **original experimental machinery** can still generate
new Gemma 4 trajectories from the official Kaggle competition inputs.

This is **not** a statistical rerun of the paper. It runs exact ALT-v1 on only
two frozen task IDs (one FastAPI and one Rich task) and checks that:

1. official task/snapshot inputs load;
2. the exact ALT-v1 agent archive validates;
3. Gemma 4 31B starts under the same vLLM configuration;
4. tool-using evaluation completes;
5. trajectory JSON and patch/output artifacts are produced;
6. the result schema matches the preserved research bundles.

Expected runtime is only two agent evaluations, not the full 30/60-run study.


In [ ]:
from pathlib import Path
import os, json, hashlib

MODEL = "gemma-4-31b-it-qat-w4a16-ct"
SMOKE_TASK_IDS = ["fastapi_14262", "rich_3942"]
N_TASKS = len(SMOKE_TASK_IDS)

WORKING_DIR = Path("/kaggle/working")
DATA_DIR = Path("/kaggle/input/competitions/gemma-4-developer-agent")
TASKS_PATH = DATA_DIR / "tasks.jsonl"
SNAPSHOTS_DIR = DATA_DIR / "snapshots"
MODEL_PATH = Path("/kaggle/input/models/google/gemma-4/other") / MODEL / "2"

STUDY_ROOT = WORKING_DIR / "Gemma4_ALT_v1_SmokeTest"
CHECKPOINT_ZIP = WORKING_DIR / "Gemma4_ALT_v1_SmokeTest.zip"
FINAL_ZIP = CHECKPOINT_ZIP

STUDY_ROOT.mkdir(parents=True, exist_ok=True)

print("ALT-v1 reproducibility smoke test")
print("Frozen tasks:", SMOKE_TASK_IDS)
print("Planned evaluations:", N_TASKS)


## 1. Install the official competition wheelhouse

In [ ]:
# Run once per runtime. Adapted from the ALT-v1 baseline notebook.
# It installs the competition's offline wheel files.

import glob
import importlib
import logging
import os
import subprocess
import sys
from pathlib import Path

# ── 1. Environment variables ─────────────────────────────────────────────
# Set these before importing transformers / vLLM / LiteLLM, because those
# libraries read them once at import time.
os.environ.update({
    # transformers: PyTorch only (skip TF/Flax), never try to download models
    "TRANSFORMERS_NO_TF": "1",
    "TRANSFORMERS_NO_FLAX": "1",
    "TRANSFORMERS_OFFLINE": "1",
    # vLLM: no telemetry, spawn worker processes (safe with CUDA in notebooks),
    # allow up to 20 min for the engine to load the model, and count CUDA graphs
    # when estimating GPU memory
    "VLLM_NO_USAGE_STATS": "1",
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
    "VLLM_ENGINE_READY_TIMEOUT_S": "1200",
    "VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS": "1",
    # Multi-GPU communication: only log warnings
    "NCCL_DEBUG": "WARN",
    # LiteLLM: use the bundled cost map (no internet fetch), no telemetry, less logging
    "LITELLM_LOCAL_MODEL_COST_MAP": "True",
    "LITELLM_TELEMETRY": "False",
    "LITELLM_LOG": "WARNING",
    # PyTorch: lower memory fragmentation on the GPU
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    # OpenTelemetry: off (it would try to export traces over the network)
    "OTEL_SDK_DISABLED": "true",
    "OTEL_PYTHON_DISABLED": "true",
})
# The OpenAI-compatible clients require some API key. The local vLLM server
# doesn't check it, so "EMPTY" is enough. setdefault keeps any real key you set.
os.environ.setdefault("OPENAI_API_KEY", "EMPTY")
os.environ.setdefault("LOCAL_API_KEY", "EMPTY")


# ── 2. Quiet noisy loggers ───────────────────────────────────────────────
NOISY_LOGGERS = [
    "LiteLLM", "LiteLLM Router", "LiteLLM Proxy", "litellm",
    "httpx", "httpcore", "openai", "urllib3",
    "opentelemetry", "opentelemetry.context",
]

def configure_logging() -> None:
    for name in NOISY_LOGGERS:
        logger = logging.getLogger(name)
        logger.setLevel(logging.WARNING)       # hide INFO/DEBUG messages
        if "litellm" in name.lower():
            logger.propagate = False           # stop LiteLLM messages appearing twice
            logger.handlers.clear()            # drop LiteLLM's own handlers

configure_logging()


# ── 3. Remove broken cutlass .pth hooks ──────────────────────────────────
# Python runs .pth files at startup. The cutlass ones in Kaggle's image can
# crash imports, so delete them if they're there.
for pth in glob.glob("/usr/local/lib/python*/*-packages/*cutlass*.pth"):
    Path(pth).unlink(missing_ok=True)


# ── 4. Restore '+cu128' in wheel filenames ───────────────────────────────
# Kaggle datasets drop the '+' from names like 'torch-2.x+cu128-...whl', and pip
# rejects a wheel whose filename doesn't match its metadata. So create correctly
# named links in /tmp. /kaggle/input is read-only, so we can't rename in place.
# Wheelhouse location varies by Kaggle attachment style; require at least one wheel.

# Robust wheelhouse discovery: Kaggle input mount paths can differ by slug/version.
WHEELHOUSE_HINTS = [
    Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse"),
    Path("/kaggle/input/gemma-4-developer-agent-wheelhouse"),
]

def _find_wheelhouse():
    # Prefer known locations first.
    for p in WHEELHOUSE_HINTS:
        if p.exists() and any(p.rglob("*.whl")):
            return p

    # Fall back to searching all attached Kaggle inputs for a directory
    # that actually contains wheel files.
    root = Path("/kaggle/input")
    if root.exists():
        dirs = {}
        for whl in root.rglob("*.whl"):
            dirs.setdefault(whl.parent, 0)
            dirs[whl.parent] += 1
        if dirs:
            # Pick the directory containing the largest number of wheels.
            return max(dirs, key=dirs.get)
    return None

WHEELHOUSE_DIR = _find_wheelhouse()

if WHEELHOUSE_DIR is None:
    attached = []
    root = Path("/kaggle/input")
    if root.exists():
        attached = sorted(str(p) for p in root.iterdir())
    raise AssertionError(
        "Gemma Developer Agent wheelhouse not found. "
        "Attach the competition wheelhouse dataset to this Kaggle notebook, then rerun. "
        f"Currently attached top-level inputs: {attached}"
    )

print("Using wheelhouse:", WHEELHOUSE_DIR)
FIXED_DIR = Path("/tmp/wheelhouse")
FIXED_DIR.mkdir(parents=True, exist_ok=True)

for whl in WHEELHOUSE_DIR.glob("*.whl"):
    if "cutlass" in whl.name.lower():          # skip the cutlass wheel (same breakage as step 3)
        continue
    name = whl.name
    if "cu128" in name and "+" not in name:    # put the missing '+' back
        name = name.replace("cu128", "+cu128")
    link = FIXED_DIR / name
    if link.is_symlink() and not link.exists():
        link.unlink()
    if not link.exists():
        link.symlink_to(whl)                   # a symlink is instant; no need to copy GBs


# ── 5. Install offline ───────────────────────────────────────────────────
# --no-deps: the wheelhouse already has every dependency, so pip never needs the internet
# --force-reinstall: replace the versions preinstalled in Kaggle's image
wheels = sorted(str(w) for w in FIXED_DIR.glob("*.whl"))
assert wheels, f"No wheels found: {WHEELHOUSE_DIR}"
print(f"Installing competition wheelhouse ({len(wheels)} wheels)...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", *wheels],
    check=True,                                # stop the notebook if the install fails
)

importlib.invalidate_caches()                  # make this process see the new packages
configure_logging()                            # newly installed packages may reset log levels
print("Wheelhouse installation complete.")

## 2. Freeze a fresh 30-task cohort

In [ ]:
import pandas as pd
from swegemma.models import load_tasks

assert TASKS_PATH.is_file(), f"Missing tasks: {TASKS_PATH}"
assert SNAPSHOTS_DIR.is_dir(), f"Missing snapshots: {SNAPSHOTS_DIR}"

tasks = load_tasks(TASKS_PATH)
tasks_by_id = {t.instance_id: t for t in tasks}

missing = [tid for tid in SMOKE_TASK_IDS if tid not in tasks_by_id]
assert not missing, f"Smoke task IDs missing from current official task catalog: {missing}"

selected = list(SMOKE_TASK_IDS)
records = []
for tid in selected:
    t = tasks_by_id[tid]
    records.append({
        "task_id": tid,
        "repo": str(t.repo),
        "base_commit": str(t.base_commit),
        "word_count": len(str(t.problem_statement).split()),
    })

cohort = {
    "purpose": "reproducibility smoke test only",
    "n_tasks": N_TASKS,
    "task_ids": selected,
    "catalog_sha256": hashlib.sha256(TASKS_PATH.read_bytes()).hexdigest(),
    "records": records,
}
(STUDY_ROOT / "cohort.json").write_text(json.dumps(cohort, indent=2) + "\n")
pd.DataFrame(records).to_csv(STUDY_ROOT / "cohort.csv", index=False)

print("Official competition inputs loaded successfully.")
display(pd.DataFrame(records))


## 3. Build the two replication agents from exact ALT-v1

In [ ]:
import base64, io, zipfile, difflib, shutil, yaml
from adk_submission import validate_directory
from adk_submission.discovery import discover_declared_models
from swegemma.config import build_submission_limits

ORIGINAL_ALT_V1_SHA256 = '277b20147e89904e1da3386515cef9bc3ab662c7775441d851a0a66405652e08'
ORIGINAL_ALT_V1_B64 = 'UEsDBBQAAAAIAAAAIQAQu8lwwwAAADoBAAAKAAAAYWdlbnQueWFtbFWQQW4DIQxF95yCHoBUUaMu5jLIAw5jCRsKRqPcPnQSJe3KevbXt78FGBfbd/SQUNRwiZgXm5AZ3MV9nVdH6n5A3X6B87cLaki6thGUiiz2gyTkEdHWVrhq/+y3rsgnjkZLyX0x1jrbhvhQpqXEByNEf6WMB2EkfdPeSPGNCdV3BR39wD5WnuoKGrbXPMyjvSClbS3tqUNoYfOdmDK0Q/FfPo1Sg7qZGRsbzJWhiM4X/NYrpT/RHo0ZDbhmknS6AWdzB1BLAwQUAAAACAAAACEAPcBnEmUAAAB3AAAAFQAAAGNvbmZpZ3Mvc2FtcGxpbmcueWFtbD2MUQrCMBAF/3OKPYFUUTG5zFLbNQm1uyG+hR7f+OPfMAwD2Zv0Gd4l0XS6BFjj9sN4C/t8sDmag2Gb6CfR4xxHU6puVTMvpq+aUyD6q6evWZDoOsX78FWXt6/CKOa5YBzQXcIXUEsDBBQAAAAIAAAAIQCBs8SDSwAAAF4AAAAQAAAAZXZhbF9jb25maWcueWFtbEstS8wpTSzJzM+z4lJQKMnMTc0vLYkvTk3Oz0sptlIwMjEACucmVsSX5OfnxCcn5uSARC1ggkD18bmZeaUlqUBhuGhpUR6Ia8AFAFBLAwQUAAAACAAAACEAVz3/XFoMAAAzGwAAEQAAAHByb21wdHMvc3lzdGVtLm1kjVnbbhtHEn33VzQSICsHJH1JAmRlYAGtLCPCKrZhx7vIE9mcaZK9nJkeT/eIYr5+z6nquUhBgH2xNeT0papOnTpV/D30xnbO2MbYPoUm1KGPJoZdOvFj1+x941y3Mr+HvjP/DdtLU9ujM+ngTKxtVbmYTBG6zhX4/2CbPb4L8vXnD18+Xd/gy9IZ35gXp9AdY2sLLOQLNslbB1+WrjEJ+0SzC5186GPsndm6KpxMa2Ncmd+evnpyuN6p8ynho10X6tnC5B7SQk+BCQ8WV2ts7eLC4J9o9/wrnVsXYXaJYw723sO8yGtOu9B+MbN1hd/5wiYfmtWzZ9+aX64+vTV3t7/e/vb52dK8/hkWh8oU8Ibu+LOpfdPjnrJhSLZamf8ccE/nsWFnur6JJvS4I/enX3yzN6lzPNrsO1u60lisTnhemSssLH1Sn7kHT/OL0DcJZrh7bOt35ow4Nnjo5BpmE/tt7dO6tak4bGSLvuEm2FjOsdXJnhFohA7X/MN1YQVT3usOnbPJwVmh9LuzQUxKV7nkxPNm5xFzc7Hhw/r7VXveLMzm+zUf9cE253SgQX1TYjd5Mb7gW+1Z3vKN51MRmt2w6jkjDNt9HJBYw92F2Vlf9Z3j3T4XHY3J5++DYCrVrQlNdaaFZ/nKVG6X8tEzyG1dERB8oKlLJuzE8dnR4qPJegTHWLPr4UWxN/bwmrnYEg7ZgskWs+LffUMY4qn0cCg2oT1X6mECD1ecOW8hGLFlaTbL48acCAyGr7DNdIsqhCMhEj2yZ7KD21YIR+tgYFN4QpjZWyFk5RkuiYk5WU4b9RHXjs52xWEdfe0r262Zkrz33iX5e904vz9sQxc3jPb0OVAEJ7VEEBLwbDqX+q4xTcgBjg5+kpS4vrq7Mx/vrt7j4GtJhFfLny9hBtLGicG84Sp/9/flq9eXAmohFvPrl8+/GWShMxmjFtnbd4ibhHR7VlS/er0yt4p1Wo0sAjYWSkhn8tMWPl7ueyS5ZkzO5zrA9ZU/uups2opwaMIJrgR/BJghr8L7JiaPU7ZuIDT40XzEEXJ3GoyXYLtmzWqy9Ifl65eXBt5mutDWvH54xbx+CQiAUMtLnIw8PSGN2yp0Sinm+uCK4wK2PoC1JHG5MgNoh3SJB3MCdSAyniDb7TagygrsCFfBOU+ynfH4+OnD9c3bL59u8PCtebUyNw+pIxH+idHMBa40EtjzZ+8ecekCfiNuAQt6OLpqdzmjVcXzrm8KbrUwRQW2Joau724F+m9mL7uuwzeZgsVP7qFwrVyCdDy824rrB7Dd2wpkjIXIhrZP+pJv8GfMVaTz+73TwrHt92/Iid0ZUcUhhQX8a9fwDGx58R7/gTTrNp0XiMMeHrjHB0hg4n1hyr6t6BdWiNLtbF+lSG4iI+Z9QVsgSIA64mBggnZIkj8uHkhMuC65zlZ6zbqHH23bIhXVHxXzVXHLw1cSKSD8TnIGAPgSB88hsIf4tMqVnhgj90lt0RzPhNC4gm7uYCV4b4sajrMG2kjTu48K7t9iPi6e622oopQnrxXFdTFnceXuLbYQOtN7+QYVnxQH6gRv1DVeXMntCdd951pFL/DtymV2G5bCOY1eYXCU5+2AzOE0rq/iUilf7goMHxFIoQU5HeG2qAqoYoiViYfQVyX/O6F6xuk92a6iH5q+3gItoF0SZyvx3laSzv+UAsN6MnnodAhxxNuca4/gkEb4iA5sIDnyokzPSAMFxM53cJVecEGKJr4VE/nWWwfbXF4umNpbj73vwNdJ7ctbw9F2i0Qgo9S8Ld+mWeqvo4OvRXDkbKEnOuSMSoOyR7mEQZXLVTRHHxEK0acAdFMc0BUfmkLVT7IdygELs3J5uRA2zOsyGHzdVo5ppozSVn1UQAwvKITkPBGMSMsM2y1qScbRGKKOMlK9LV6ejoxZdeKDPU4SgfUEeahKdEoESgR1ZFpfEEjT1gvehJWR+64FJEQXqg6WVi7vjgrha/1CfP7jy+zqljJBXtFSnrTCpAOV1XAzvjQmw21TVH1JPR36PaLZgwqhUaSmUB8jriJZUL6xYfT7xoL85iykfCkMCpVd4iJ7xumdXBu0d8lCUWZYi6yU2jaKW0aw7ULZFyyv4iZ970loHt9EXgiNlo4MU5bLR6fvnFw230CWVEiamMWqRMNveWNzQReLCJkVjdDOioeEpgwFqYioep4LanuWhM60wTyiR7knHQ5SydALYCPo2h7JAg5er2Hdej2cIcWMTvVRaiXQva1y7gDDSJVIsz45OmcCpThGOogx+welR4nwNhhRBcx6vB9E448VldkFB0laj2nM5B+2sk1kM6Oe+wrxon3GcqAWZXu1lzdQxMTGo5aMPJJDG2UFHmBWvRClVNuzaFpkQGBSS1JAvR7gjmVhLpg+vOdi6B9E/TOsnt4HHqJzOd3BKwJBrVU/QFUAC0Olwp9TKmkrGLl8BIFUOXPNUIaqXOdCMNTCsbqp8ptH+HRAfRD9q9q5OASSslV2Y/WG25QPte6wL3R6GWFzTe69a3rkL56pwOYCc+xG1ded+9qjuMYchMkyEiUwQhFXa62AnkF2egBVP2HKjvyYq6keNCMMyAdQrQTC5i66G6zIcV0N3YiAh02R1kZESpoiwEXl8xV6KKDtFKZ6kJsm3IQwDlL7hfw+atRj0flW6ErIaM6d+U5f+0AFRi5BKr5BJkXXpclNCJ802SYUBfvmIYYCsEwTuDmkNu/jk1jNoEife1CvQ1vjkq67p5JHpWBxEbKhQuGCUvOqcxLOcZ0ipXY16pWQkH/IRAZzF7JEYHSGygv1PD0Lb6ulSMIcHRLME1klml3nGAKd019Aw/yLtXZGcH+mbAXrFoqF0p/FAJTitx5q5wyTK7Yp086RYl+44+RZHX9VgqIl6UyVDQIa9KYk4XKoA4CGD3Uc2tqnxWVGJZE6dSoJBysNMfFXPanh6PDjuSm0vslf97bzVmYO15UfQnTFr/QZrQsRC3zSYhVAItggeJg0MnLhnUibVbClZMoS5VlSCoV8s16j10nrNScC0ohm+t48X4hSO2ucxymTCnICXE1HsH+Zz4jCjsMh9+C6wrMPCFIarUiGVaatWXsyNiPaxWiDwm+GlkUg/wQuj1laIAOWRoZtMxlLrGjqUHDRzEtsJKy8ee6+e6XDeZMkDPQLw+z+1NWQ21nOkDSerSvQ+GqoyUjafgzlUhWGjUcElUw4AP+djenq4y2cm8dL9IQreimKWmXyCAf7xXXsihcbZf4fV+bf2u1eDOLntc6/nktCZhkwjjxy96r6QEKzlJbpm++P7kx/cI70zUYajkaq1Sb52lF0vfrp5VitapOnLjpSYoF+IWOoByw3ywez/GqWR5P33Ggt2OjqH7h8HNTIBitdu5HyMH6F4vlfCHRx/bvMpMp8nCBl8kG0hWn9Q8754ZPSJisSRuWEhjPOCA6w1ZZPUUyN20oaKJEAT/tGcUo4j0XA+m4AtWp219x7KDxGOSe4VX//xUEsCvPaREnnmKkyd5gVXiHxRa5TDAWfBeacE6h6lxgtSNBMvs2oJjY4zRVHyd150FhfWuBgM/REecwjZUAQ9dOKsAF56AaigmWoQTlGPLAVhCZN6CyWS637331npnlIbgQIwtV8TmIrfnzwlE59M+8f4Y0tAPZ4eDLpLA1AHjdRAtlmtkGcTSONqAv1czmf83WoUahuVF4MYkwMLJ6mWY5YS5gvl5uxDZT7KBiuQ4P2sRb8XebOdNItcTh0mHkwqRaPJiycqyhmlSHyBGIs2W+mYd4I0ZOlboOC5sZJ+2TOFILMVoN0kg4tB4ArYhvGUX/r/DWOQ6+nM+jBlo18sI7+D2f+YV5u5J4bWbsenMjPxfl5Yo7m5aXKWZkSsp4Pk8I8bVYtysHeDDy5kcaNbiZxoz3eMjoqDf4a0UP7sI7sssR4kAna25vr28+3H96bT1/ubjjnv6q3ft/z5xHh+stBhs4mjDrFGKaFYV4ZpHZINS0oxA21W1YmYOZR98e58pFhrsy+c2VFm85OymkvztcvTeuRLyLKgPP/q/1TjaSvyk8MQExHWRy2IsdKPVI6FR3Cwc8ZJpcm5ZLL9MKyJo3jxDwRyywuNWWcypI0Sl/qMPh8sqLcPqeeuS4K1g6yFv7PvzHZ6YzFKJMVIaPvsj4exY2I4umXjJD7buHvAemNc6V0JsNeN5I8rJCjvsPhDUqnTAqHlMs/FPwPUEsBAhQDFAAAAAgAAAAhABC7yXDDAAAAOgEAAAoAAAAAAAAAAAAAAKSBAAAAAGFnZW50LnlhbWxQSwECFAMUAAAACAAAACEAPcBnEmUAAAB3AAAAFQAAAAAAAAAAAAAApIHrAAAAY29uZmlncy9zYW1wbGluZy55YW1sUEsBAhQDFAAAAAgAAAAhAIGzxINLAAAAXgAAABAAAAAAAAAAAAAAAKSBgwEAAGV2YWxfY29uZmlnLnlhbWxQSwECFAMUAAAACAAAACEAVz3/XFoMAAAzGwAAEQAAAAAAAAAAAAAApIH8AQAAcHJvbXB0cy9zeXN0ZW0ubWRQSwUGAAAAAAQABAD4AAAAhQ4AAAAA'

raw = base64.b64decode(ORIGINAL_ALT_V1_B64)
assert hashlib.sha256(raw).hexdigest() == ORIGINAL_ALT_V1_SHA256
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert z.testzip() is None
    baseline_files = {n:z.read(n) for n in z.namelist()}

assert set(baseline_files) == {
    "agent.yaml", "eval_config.yaml", "configs/sampling.yaml", "prompts/system.md"
}
base_prompt = baseline_files["prompts/system.md"].decode()

PROMPTS = {"A_alt_v1": base_prompt}

AGENT_ROOT = STUDY_ROOT / "agents"
AGENT_DIRS = {}
limits, generation_constraints = build_submission_limits()

for label, prompt in PROMPTS.items():
    folder = AGENT_ROOT / label
    if folder.exists():
        shutil.rmtree(folder)
    for name, blob in baseline_files.items():
        target = folder / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(prompt.encode() if name == "prompts/system.md" else blob)
    validate_directory(folder, limits=limits)
    assert discover_declared_models(folder) == {MODEL}
    AGENT_DIRS[label] = folder

# Exact-baseline integrity checks.
for name in ["agent.yaml", "eval_config.yaml", "configs/sampling.yaml", "prompts/system.md"]:
    assert (AGENT_DIRS["A_alt_v1"] / name).read_bytes() == baseline_files[name]

study_plan = {
    "purpose": "2-task reproducibility smoke test",
    "condition": "A_alt_v1 exact original baseline",
    "shared_config": {
        "model": MODEL,
        "max_tool_calls": 28,
        "max_time_minutes": 8,
        "max_turns": 80,
        "thinking_budget": 4096,
        "temperature": 0.2,
        "declared_tools": 9,
    },
    "baseline_zip_sha256": ORIGINAL_ALT_V1_SHA256,
}
(STUDY_ROOT / "experiment_plan.json").write_text(json.dumps(study_plan, indent=2) + "\n")

print("Exact ALT-v1 agent prepared.")
print("Baseline archive SHA256:", ORIGINAL_ALT_V1_SHA256)


## 4. Start Gemma 4 31B once

In [ ]:
import os, requests, torch, litellm
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

assert torch.cuda.device_count()==4, 'Choose Kaggle 4 × L4 GPUs'
assert MODEL_PATH.is_dir(), f'Attach model version 2: {MODEL_PATH}'
litellm.drop_params=True
adapters=discover_adapters(str(AGENT_DIRS['A_alt_v1']),adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
assert not (getattr(adapters,'adapters',None) or {}), 'Original agent should not contain adapters'

# Reuse the server if just running the next batch within the same Kaggle session.
ready=False
signature=(str(MODEL_PATH),4,32768,0.80)
if globals().get('server_instance') is not None and globals().get('SERVER_SIGNATURE')==signature:
    try: ready=requests.get(server_instance.base_url.rstrip('/')+'/models',timeout=5).ok
    except Exception: pass
if not ready:
    if globals().get('server_instance') is not None:
        try: server_instance.stop()
        except Exception as exc: print('Old server stop warning:',exc)
    config=VllmConfig(
        model=str(MODEL_PATH),host='127.0.0.1',port=8000,
        tool_call_parser='gemma4',reasoning_parser='gemma4',enable_auto_tool_choice=True,
        max_model_len=32768,dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'auto',
        gpu_memory_utilization=0.80,enable_lora=False,tensor_parallel_size=4,
        scheduler_reserve_full_isl=False,startup_timeout=20*60)
    server_instance=VllmServer(config,adapter_manifest=adapters)
    server_instance.start()
    SERVER_SIGNATURE=signature
models=server_instance.create_model_registry(aliases=[MODEL],model_prefix='openai/',
                                           api_key=os.environ.get('LOCAL_API_KEY','EMPTY'))
requests.get(server_instance.base_url.rstrip('/')+'/models',timeout=15).raise_for_status()
print('Gemma inference ready:',server_instance.base_url)

## 5. Run exact ALT-v1 on the two frozen smoke-test tasks

In [ ]:
import asyncio, concurrent.futures, contextlib, io, time, traceback, zipfile, shutil, math
import pandas as pd
from collections import Counter
from IPython.display import display, clear_output, FileLink
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig
from swegemma.evaluate import Evaluator

CONDITIONS = ["A_alt_v1"]
TOTAL_RUNS = N_TASKS * len(CONDITIONS)
RESULTS_JSON = STUDY_ROOT/"results.json"

# Separate evaluator output trees avoid trace collisions.
evaluators={}
for label in CONDITIONS:
    evaluators[label]=Evaluator(EvalConfig(
        tasks_path=TASKS_PATH,
        snapshots_dir=SNAPSHOTS_DIR,
        submission_dir=AGENT_DIRS[label],
        results_dir=STUDY_ROOT/"evaluation"/label,
        wheels_dir=DATA_DIR/"wheels",
        graph_dir=str(DATA_DIR/"graphs"),
        embeddings_dir=str(DATA_DIR/"embeddings"),
        models=models,
        adapter_manifest=adapters,
        sandbox="subprocess",
        timeout_seconds=240,
        verification_timeout_seconds=300,
        max_tool_calls=28,
        max_time_minutes=8,
        max_turns=80,
        limits=limits,
        generation_constraints=generation_constraints,
        context_cache_config=ContextCacheConfig(
            min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=15, overlap_size=2,
            token_threshold=14336, event_retention_size=5),
        verbose=False,
    ))

def async_run(evaluator, task, task_index):
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(lambda: asyncio.run(
            evaluator.evaluate_task(task=task, task_index=task_index, total_tasks=TOTAL_RUNS)
        )).result()

def trace_metrics(trace_path):
    out={
        "trace_available":False,
        "trace_tool_calls":None,
        "first_successful_edit_call":None,
        "read_calls":None,
        "run_command_calls":None,
        "edit_calls":None,
        "write_calls":None,
        "graph_calls":None,
        "tool_error_count":None,
    }
    if not trace_path.exists():
        return out
    try:
        trace=json.loads(trace_path.read_text())
        steps=trace.get("steps",[])
    except Exception:
        return out
    out["trace_available"]=True
    call_index=0
    counts=Counter()
    errors=0
    first_edit=None
    for step in steps:
        if step.get("source")!="agent":
            continue
        obs=step.get("observation") or {}
        obs_text=json.dumps(obs, default=str).lower()
        step_error=('"status": "error"' in obs_text) or ("error_type" in obs_text and '"error_type": null' not in obs_text)
        for call in step.get("tool_calls") or []:
            call_index+=1
            name=call.get("function_name","")
            counts[name]+=1
            if step_error:
                errors+=1
            if first_edit is None and name in {"edit_file","write_file"} and not step_error:
                first_edit=call_index
    out.update({
        "trace_tool_calls":call_index,
        "first_successful_edit_call":first_edit,
        "read_calls":counts["read_file"],
        "run_command_calls":counts["run_command"],
        "edit_calls":counts["edit_file"],
        "write_calls":counts["write_file"],
        "graph_calls":sum(counts[x] for x in ["get_code_neighbors","search_similar_code","get_code_subgraph"]),
        "tool_error_count":errors,
    })
    return out

def checkpoint_zip():
    tmp=CHECKPOINT_ZIP.with_suffix(".zip.tmp")
    with zipfile.ZipFile(tmp,"w",zipfile.ZIP_DEFLATED,compresslevel=1) as z:
        for p in sorted(STUDY_ROOT.rglob("*")):
            if p.is_file() and not p.is_symlink():
                z.write(p,p.relative_to(WORKING_DIR).as_posix())
    os.replace(tmp,CHECKPOINT_ZIP)

def compact_screen(results, current=None):
    clear_output(wait=True)
    df=pd.DataFrame(results)
    print(f"Gemma4 ALT-v1 Smoke Test — {len(results)}/{TOTAL_RUNS} executions complete")
    if current:
        print("Current:", current)
    if not df.empty:
        summary=(df.groupby("condition")
                   .agg(n=("task_id","count"),
                        resolved=("resolved_local",lambda x: int(pd.Series(x).fillna(False).sum())),
                        empty_patch=("patch_chars",lambda x: int((pd.Series(x)==0).sum())),
                        median_calls=("tool_calls","median"),
                        median_s=("elapsed_seconds","median"))
                   .reset_index())
        display(summary)
        display(df[["task_id","repo","condition","status","tool_calls","elapsed_seconds","patch_chars"]].tail(8))
    print("Full stdout/stderr, traces, patches and verifier artifacts are being written under:")
    print(STUDY_ROOT)
    print("Checkpoint ZIP refreshes every 10 executions:", CHECKPOINT_ZIP)

results=[]
completed=set()
if RESULTS_JSON.exists():
    results=json.loads(RESULTS_JSON.read_text())
    completed={(r["task_id"],r["condition"]) for r in results}

# Counterbalance condition order by task to reduce systematic ordering bias.
schedule=[]
for i,tid in enumerate(selected):
    shift=i % len(CONDITIONS)
    order=CONDITIONS[shift:]+CONDITIONS[:shift]
    for label in order:
        schedule.append((tid,label))
assert len(schedule)==TOTAL_RUNS
assert len(set(schedule))==TOTAL_RUNS
(STUDY_ROOT/"execution_schedule.json").write_text(json.dumps(schedule,indent=2)+"\n")

artifact_root=STUDY_ROOT/"captured_output"
artifact_root.mkdir(exist_ok=True)
compact_screen(results)

for run_index,(tid,label) in enumerate(schedule,1):
    if (tid,label) in completed:
        continue
    task=tasks_by_id[tid]
    current=f"{run_index}/{TOTAL_RUNS}  {label}  {tid}"
    compact_screen(results,current=current)

    cond_art=artifact_root/label
    cond_art.mkdir(parents=True,exist_ok=True)
    stdout,stderr=io.StringIO(),io.StringIO()
    start=time.monotonic()
    outcome=None
    patch=""
    try:
        with contextlib.redirect_stdout(stdout), contextlib.redirect_stderr(stderr):
            outcome=async_run(evaluators[label],task,run_index)
        patch=str(getattr(outcome,"agent_patch","") or "")
        resolved_raw=getattr(outcome,"resolved",None)
        resolved=None if resolved_raw is None else bool(resolved_raw)
        test_exit=getattr(outcome,"test_exit_code",None)
        errtxt=str(getattr(outcome,"error","") or "")
        status=("resolved_local" if resolved is True else
                "not_resolved_or_verification_blocked" if resolved is False else
                "unknown")
        row={
            "task_id":tid,"repo":str(task.repo),"condition":label,
            "status":status,"resolved_local":resolved,
            "tool_calls":getattr(outcome,"tool_calls",None),
            "elapsed_seconds":round(time.monotonic()-start,1),
            "test_exit_code":test_exit,"patch_chars":len(patch),
            "error":errtxt,
        }
    except Exception as exc:
        row={
            "task_id":tid,"repo":str(task.repo),"condition":label,
            "status":"execution_error","resolved_local":None,
            "tool_calls":None,"elapsed_seconds":round(time.monotonic()-start,1),
            "test_exit_code":None,"patch_chars":0,
            "error":f"{type(exc).__name__}: {exc}",
        }
        (cond_art/f"{tid}.exception.txt").write_text(traceback.format_exc())

    (cond_art/f"{tid}.patch").write_text(patch)
    (cond_art/f"{tid}.stdout.txt").write_text(stdout.getvalue())
    (cond_art/f"{tid}.stderr.txt").write_text(stderr.getvalue())

    eval_dir=STUDY_ROOT/"evaluation"/label
    trace_path=eval_dir/"traces"/f"trace_{tid}.json"
    row.update(trace_metrics(trace_path))

    results.append(row)
    completed.add((tid,label))
    RESULTS_JSON.write_text(json.dumps(results,indent=2,default=str)+"\n")
    pd.DataFrame(results).to_csv(STUDY_ROOT/"results.csv",index=False)

    # Small progress state for easy recovery/analysis.
    progress={
        "completed":len(results),"total":TOTAL_RUNS,
        "last_task":tid,"last_condition":label,
        "updated_utc":pd.Timestamp.utcnow().isoformat(),
    }
    (STUDY_ROOT/"progress.json").write_text(json.dumps(progress,indent=2)+"\n")

    checkpoint_zip()

compact_screen(results,current="RUN COMPLETE")
checkpoint_zip()
print("All scheduled executions finished.")
display(FileLink(str(CHECKPOINT_ZIP)))

## 6. Validate smoke-test artifacts

In [ ]:
import pandas as pd, json
from pathlib import Path

df = pd.DataFrame(results)
assert len(df) == N_TASKS, f"Expected {N_TASKS} results, got {len(df)}"
assert set(df["condition"]) == {"A_alt_v1"}
assert set(df["task_id"]) == set(SMOKE_TASK_IDS)

required_columns = {
    "task_id", "repo", "condition", "status", "resolved_local",
    "tool_calls", "elapsed_seconds", "test_exit_code", "patch_chars",
    "trace_available", "trace_tool_calls", "read_calls", "run_command_calls",
    "edit_calls", "write_calls", "graph_calls", "tool_error_count",
}
missing_cols = sorted(required_columns - set(df.columns))
assert not missing_cols, f"Missing result columns: {missing_cols}"

# A smoke test is about machinery, not whether these stochastic runs solve the issues.
for tid in SMOKE_TASK_IDS:
    trace = STUDY_ROOT / "evaluation" / "A_alt_v1" / "traces" / f"trace_{tid}.json"
    assert trace.exists(), f"Missing trajectory JSON: {trace}"
    data = json.loads(trace.read_text())
    assert isinstance(data.get("steps"), list), f"Malformed trajectory: {trace}"

    patch = STUDY_ROOT / "captured_output" / "A_alt_v1" / f"{tid}.patch"
    stdout = STUDY_ROOT / "captured_output" / "A_alt_v1" / f"{tid}.stdout.txt"
    stderr = STUDY_ROOT / "captured_output" / "A_alt_v1" / f"{tid}.stderr.txt"
    for artifact in [patch, stdout, stderr]:
        assert artifact.exists(), f"Missing captured artifact: {artifact}"

df.to_csv(STUDY_ROOT / "results.csv", index=False)

print("Smoke-test result schema:")
display(df[[
    "task_id", "repo", "status", "resolved_local", "tool_calls",
    "trace_tool_calls", "patch_chars", "elapsed_seconds"
]])

print("\nSMOKE TEST PASSED:")
print("- official task/snapshot inputs loaded")
print("- exact ALT-v1 archive validated")
print("- Gemma inference server responded")
print("- both evaluations produced trajectory JSON")
print("- patch/stdout/stderr artifacts exist")
print("- result schema matches the research pipeline")
print("\nStochastic resolution outcomes are NOT expected to match historical runs bit-for-bit.")


## 7. Export smoke-test bundle

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import display, FileLink
import os, json, hashlib

manifest = {
    "study": "Gemma4_ALT_v1_SmokeTest",
    "purpose": "experimental-machinery reproducibility smoke test",
    "n_tasks": N_TASKS,
    "task_ids": SMOKE_TASK_IDS,
    "completed_runs": len(results),
    "baseline_zip_sha256": ORIGINAL_ALT_V1_SHA256,
    "cohort_sha256": hashlib.sha256((STUDY_ROOT / "cohort.json").read_bytes()).hexdigest(),
}

(STUDY_ROOT / "study_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")

tmp = FINAL_ZIP.with_suffix(".zip.tmp")
with ZipFile(tmp, "w", ZIP_DEFLATED, compresslevel=2) as z:
    for p in sorted(STUDY_ROOT.rglob("*")):
        if p.is_file() and not p.is_symlink():
            z.write(p, p.relative_to(WORKING_DIR).as_posix())
os.replace(tmp, FINAL_ZIP)

with ZipFile(FINAL_ZIP) as z:
    assert z.testzip() is None

print("Smoke-test research bundle:", FINAL_ZIP)
print("Size MB:", round(FINAL_ZIP.stat().st_size / 1e6, 2))
display(FileLink(str(FINAL_ZIP)))


## Outputs

The run writes `Gemma4_ALT_v1_SmokeTest.zip` to `/kaggle/working/`, containing:
- `cohort.csv` / `cohort.json`
- `experiment_plan.json` / `execution_schedule.json`
- `results.csv` / `results.json`
- `study_manifest.json`
- `prompts/system.md`
- per-task evaluation traces
- `captured_output/` patches and host logs

The notebook intentionally keeps the screen compact. Full logs are stored in the ZIP.